# 02. Feature Engineering & Xử lý mất cân bằng lớp

Ngày 3: Chuẩn hoá pipeline feature engineering, tách train/test, xử lý mất cân bằng lớp, và lưu dữ liệu đã xử lý vào `data/processed/` để
notebook `03_modeling.ipynb` dùng lại mà không cần lặp lại logic xử lý dữ liệu.

## 0. Mục tiêu notebook

- Chuyển các bước làm sạch/biến đổi dữ liệu đã khám phá ở `01_eda.ipynb` thành pipeline
  tái sử dụng (`src/features.py`).
- Encode toàn bộ feature về dạng số, sẵn sàng cho modeling.
- Tách 3 tập **train / validation / test** theo  **stratified** (giữ nguyên tỷ lệ
  churn ở cả 3 tập). Validation dùng để chọn model/threshold ở `03_modeling.ipynb`, còn
  test chỉ chạm đúng 1 lần ở bước đánh giá cuối cùng, tránh dùng chung 1 tập vừa để chọn
  model, chọn threshold, vừa để báo cáo kết quả (lỗi phương pháp luận phổ biến).
- Xử lý mất cân bằng lớp (churn chỉ chiếm ~26,5%), so sánh
  2 hướng tiếp cận: `class_weight='balanced'` (áp dụng trực tiếp trong model, không đụng
  vào dữ liệu) vs **SMOTENC** (oversampling tổng hợp, chỉ áp dụng trên tập train để tránh
  data leakage sang validation/test).
- Lưu kết quả vào `data/processed/`.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import features as fe

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
PROCESSED_DIR = '../data/processed'

## 1. Chạy pipeline làm sạch + feature engineering

Toàn bộ logic (impute `TotalCharges`, gộp "No internet/phone service" -> "No", tạo
`tenure_bucket`/`NumAddonServices`/`AvgMonthlySpend`).

In [2]:
df_raw = fe.load_raw_data(DATA_PATH)
df = fe.clean_data(df_raw)
df = fe.consolidate_service_columns(df)
df = fe.engineer_features(df)

print(df.shape)
df.head()

(7043, 23)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,NumAddonServices,AvgMonthlySpend
customerID,,,,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-6,1,29.850000
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48,2,55.573529
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-6,2,54.075000
7795-CFOCW,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48,3,40.905556
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-6,0,75.825000


In [3]:
# Kiểm tra nhanh: không còn missing, các cột add-on đã gộp về Yes/No thuần
print('Missing values:', df.isna().sum().sum())
print()
print('OnlineSecurity value_counts (đã gộp "No internet service" -> "No"):')
print(df['OnlineSecurity'].value_counts())
print()
print('Feature phái sinh:')
df[['tenure', 'tenure_bucket', 'NumAddonServices', 'AvgMonthlySpend']].describe(include='all')

Missing values: 0

OnlineSecurity value_counts (đã gộp "No internet service" -> "No"):
OnlineSecurity
No     5024
Yes    2019
Name: count, dtype: int64

Feature phái sinh:


,tenure,tenure_bucket,NumAddonServices,AvgMonthlySpend
count,7043.000000,7043,7043.000000,7043.000000
unique,NaN,5,NaN,NaN
top,NaN,49-72,NaN,NaN
freq,NaN,2239,NaN,NaN
mean,32.371149,NaN,2.037910,64.698218
std,24.559481,NaN,1.847682,30.270670
min,0.000000,NaN,0.000000,0.000000
25%,9.000000,NaN,0.000000,35.649000
50%,29.000000,NaN,2.000000,70.300000
75%,55.000000,NaN,3.000000,90.174158


## 2. Encode target & feature matrix

- `encode_target`: `Churn` (Yes/No) -> 0/1.
- `encode_features`: binary Yes/No -> 0/1, `gender` -> 0/1 (1 = Female), one-hot cho các
  cột nominal nhiều hạng mục (`InternetService`, `Contract`, `PaymentMethod`,
  `tenure_bucket`), bỏ cột `Churn` (nhãn xử lý riêng).

**Lưu ý:** `gender` và `PhoneService` được **giữ lại** trong X dù insight từ EDA (Cramér's V,
chi-square) cho thấy không có ý nghĩa thống kê (p>0,05). Quyết định loại bỏ hay không sẽ
để ở bước modeling (thử nghiệm feature selection / so sánh hiệu năng có-không), không cắt
bỏ sớm ở bước feature engineering.

In [4]:
y = fe.encode_target(df)
X = fe.encode_features(df)

print('X shape:', X.shape)
print('y shape:', y.shape, '| tỷ lệ churn:', round(y.mean(), 4))
print()
print(X.dtypes.value_counts())
X.head()

X shape: (7043, 29)
y shape: (7043,) | tỷ lệ churn: 0.2654

int64      26
float64     3
Name: count, dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,MonthlyCharges,TotalCharges,NumAddonServices,AvgMonthlySpend,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bucket_7-12,tenure_bucket_13-24,tenure_bucket_25-48,tenure_bucket_49-72
customerID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,1,0,1,0,1,0,0,0,1,0,0,0,0,1,29.85,29.85,1,29.850000,0,0,0,0,0,1,0,0,0,0,0
5575-GNVDE,0,0,0,0,34,1,0,1,0,1,0,0,0,0,56.95,1889.50,2,55.573529,0,0,1,0,0,0,1,0,0,1,0
3668-QPYBK,0,0,0,0,2,1,0,1,1,0,0,0,0,1,53.85,108.15,2,54.075000,0,0,0,0,0,0,1,0,0,0,0
7795-CFOCW,0,0,0,0,45,0,0,1,0,1,1,0,0,0,42.30,1840.75,3,40.905556,0,0,1,0,0,0,0,0,0,1,0
9237-HQITU,1,0,0,0,2,1,0,0,0,0,0,0,0,1,70.70,151.65,0,75.825000,1,0,0,0,0,1,0,0,0,0,0


## 3. Tách train/validation/test (stratified)

Tách 3 tập theo tỷ lệ **60/20/20**, `stratify=y` ở cả 2 lần tách để giữ nguyên tỷ lệ
churn ~26,5% ở cả 3 tập. Thực hiện qua 2 bước `train_test_split`: tách test 20% trước,
sau đó tách phần còn lại (80%) thành train (75% của 80% = 60% tổng) và validation (25%
của 80% = 20% tổng).

**Vì sao cần thêm validation:** nếu chỉ có train/test, việc *chọn model tốt nhất* và
*chọn threshold tối ưu theo chi phí* đều phải đánh giá trên test set, trong khi test set
cũng chính là tập dùng để *báo cáo kết quả cuối cùng*. Dùng chung 1 tập cho cả chọn lựa
lẫn báo cáo khiến con số cuối (ROC-AUC, mức tiết kiệm chi phí) lạc quan hơn thực tế, một
dạng leakage nhẹ vì đã "nhìn thấy" test set khi ra quyết định. Vì vậy mọi lựa chọn (model,
hyperparameter, threshold) đều dựa trên validation; test set chỉ dùng đúng 1 lần để xác
nhận kết quả cuối cùng, không tham gia vào bất kỳ quyết định nào.

In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=RANDOM_STATE
)

print('Train      :', X_train.shape, '| churn rate:', round(y_train.mean(), 4))
print('Validation :', X_val.shape, '| churn rate:', round(y_val.mean(), 4))
print('Test       :', X_test.shape, '| churn rate:', round(y_test.mean(), 4))

Train      : (4225, 29) | churn rate: 0.2653
Validation : (1409, 29) | churn rate: 0.2654
Test       : (1409, 29) | churn rate: 0.2654


## 4. Xử lý mất cân bằng lớp

Churn chỉ chiếm ~26,5% -> model dễ thiên vị lớp đa số (dự đoán "không churn" cho hầu hết
khách hàng vẫn đạt accuracy cao nhưng vô dụng về mặt kinh doanh vì bỏ sót đúng nhóm cần
can thiệp). Hai hướng xử lý sẽ được so sánh ở `03_modeling.ipynb`:

1. **`class_weight='balanced'`**: không đụng vào dữ liệu, chỉ tăng trọng số lỗi sai ở lớp
   thiểu số khi train. Áp dụng trực tiếp trong model (Logistic Regression, Random Forest,
   XGBoost, LightGBM đều hỗ trợ tham số này/tương đương).
2. **SMOTENC** (SMOTE for Nominal and Continuous): sinh thêm mẫu tổng hợp cho lớp churn,
   cân bằng phân phối lớp trước khi train.

**Vì sao SMOTENC chứ không phải SMOTE thường:**
- X đã encode có phần lớn là cột categorical/binary (one-hot, Yes/No -> 0/1). `SMOTE`
  thường nội suy tuyến tính trên MỌI cột kể cả cột one-hot, tạo ra giá trị phân số phi
  thực tế (vd. `Contract_Two year = 0.3`), và khi ép kiểu kết quả về lại dtype gốc (int),
  imbalanced-learn truncate các giá trị phân số này về 0 một cách có hệ thống, khiến mẫu
  tổng hợp bị thiên lệch ngầm về phía "0" cho mọi cột one-hot, sai lệch phân phối
  categorical thật của dữ liệu.
- `SMOTENC` xử lý đúng hơn: chỉ nội suy tuyến tính trên cột continuous (`tenure`,
  `MonthlyCharges`, `TotalCharges`, `AvgMonthlySpend`, `NumAddonServices`), còn cột
  categorical thì lấy **mode của k-neighbors**, không tạo giá trị phân số phi thực tế.

**Quan trọng:** SMOTENC chỉ được áp dụng lên **tập train**, và áp dụng trực tiếp bên trong
pipeline huấn luyện của mỗi fold cross-validation ở `03_modeling.ipynb.

## 5. Lưu dữ liệu đã xử lý

Lưu vào `data/processed/`:
- `train.csv`: tập train gốc (chưa resample). Dùng trực tiếp với `class_weight='balanced'`,
  hoặc làm đầu vào cho SMOTENC áp dụng bên trong pipeline huấn luyện của
  `03_modeling.ipynb`.
- `val.csv`: tập validation (giữ nguyên phân phối gốc, không resample), dùng để **chọn
  model, chọn hyperparameter, chọn threshold** ở `03_modeling.ipynb`.
- `test.csv`: tập test (giữ nguyên phân phối gốc, không resample), chỉ dùng
  để **xác nhận kết quả cuối cùng**.

**Giữ lại `customerID`:** cả 3 file đều lưu `customerID` làm cột đầu (không dùng
`index=False`), cần thiết để tra cứu/giải thích SHAP theo từng khách hàng cụ thể ở
`04_explainability.ipynb`.

In [6]:
train_df = X_train.copy()
train_df['Churn'] = y_train.values

val_df = X_val.copy()
val_df['Churn'] = y_val.values

test_df = X_test.copy()
test_df['Churn'] = y_test.values

train_df.to_csv(f'{PROCESSED_DIR}/train.csv', index=True)
val_df.to_csv(f'{PROCESSED_DIR}/val.csv', index=True)
test_df.to_csv(f'{PROCESSED_DIR}/test.csv', index=True)

print('Đã lưu (đều giữ cột customerID):')
print('-', f'{PROCESSED_DIR}/train.csv', train_df.shape)
print('-', f'{PROCESSED_DIR}/val.csv', val_df.shape)
print('-', f'{PROCESSED_DIR}/test.csv', test_df.shape)

Đã lưu (đều giữ cột customerID):
- ../data/processed/train.csv (4225, 30)
- ../data/processed/val.csv (1409, 30)
- ../data/processed/test.csv (1409, 30)


## 6. Tóm tắt & bước tiếp theo

- Pipeline feature engineering đã chuẩn hoá, tái sử dụng được
  cho `03_modeling.ipynb` và `04_explainability.ipynb` (đặc biệt là `df_readable`, bản
  chưa encode, cần cho việc giải thích SHAP theo từng khách hàng cụ thể).
- Feature matrix cuối cùng: **29 cột** (sau one-hot).
- Tách **train/validation/test** stratified theo tỷ lệ **60/20/20**, tỷ lệ churn giữ
  nguyên ~26,5% ở cả 3 tập. Validation dùng để chọn model/hyperparameter/threshold, test
  chỉ ở bước đánh giá cuối cùng.
- Xác định 2 phương án xử lý mất cân bằng lớp sẽ so sánh ở bước modeling
  (`class_weight` vs SMOTENC); quyết định phương án nào tốt hơn dựa trên kết quả đánh giá
  trên **validation set** ở `03_modeling.ipynb`, nơi SMOTENC được áp dụng trực tiếp bên
  trong pipeline huấn luyện (không resample sẵn thành một tập dữ liệu tĩnh ở đây).
- Dùng **SMOTENC** thay vì SMOTE thường vì X có nhiều cột categorical/one-hot: SMOTE
  thường nội suy tuyến tính cả cột one-hot rồi bị truncate sai lệch về 0 khi ép kiểu
  dtype gốc, còn SMOTENC lấy mode của k-neighbors cho cột categorical nên tránh được lỗi này.
- Cả 3 file `data/processed/` (`train.csv`, `val.csv`, `test.csv`) đều giữ cột
  `customerID`, cần thiết cho việc tra cứu khách hàng cụ thể ở bước SHAP.


**Tiếp theo:** `03_modeling.ipynb` sẽ train Logistic Regression, Random
Forest, XGBoost/LightGBM trên cả 2 phương án xử lý mất cân bằng lớp (`class_weight` vs
SMOTENC áp dụng trực tiếp bên trong pipeline, kèm tune nhẹ hyperparameter cho 3 model
tree-based để so sánh công bằng hơn), chọn model/threshold dựa trên **validation
set**, xây ma trận chi phí giữ chân vs mất khách hàng, và  đánh giá trên
**test set** để xác nhận kết quả cuối cùng.